# Evaluacion del agente entrenado

Carga el modelo de competencia, corre los episodios de evaluacion con politica greedy, reporta el puntaje promedio y el maximo, y genera un video del agente jugando. Pensado para correr celda por celda el dia de la presentacion, sin pasos manuales.

Reutiliza las funciones de `src/evaluar.py` (que a su vez reusan `src/ale_utils.py`): `cargar_modelo`, `evaluar`, `generar_video` y `predecir` (politica greedy, `deterministic=True`).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import json
import numpy as np
from IPython.display import Video

import evaluar

## Configuracion

Modelo de competencia por defecto. Se puede apuntar `RUTA_MODELO` a cualquier otro run con `config.json` junto al `.zip`.

In [ ]:
RUTA_MODELO = "../modelos/competencia/modelo_competencia.zip"
N_EPISODIOS = 5
SEMILLA = 0
EPISODIOS_VIDEO = 3

run_id = os.path.basename(os.path.dirname(RUTA_MODELO))
carpeta_video = f"../modelos/videos/{run_id}"
print(f"modelo: {RUTA_MODELO}  (run_id={run_id})")
print(f"episodios de evaluacion: {N_EPISODIOS}  |  episodios de video: {EPISODIOS_VIDEO}")
print(f"carpeta de video: {carpeta_video}")

## Cargar modelo

`cargar_modelo` lee `config.json` junto al `.zip` para reconstruir la arquitectura (extractor, dueling, algoritmo) y los pesos.

In [ ]:
modelo = evaluar.cargar_modelo(RUTA_MODELO)
config = evaluar.leer_config(RUTA_MODELO)
escala_grises = config.get("escala_grises", True)

print(f"algoritmo: {config.get('algoritmo')}")
print(f"backbone: {config.get('backbone')}")
print(f"dueling: {config.get('dueling', False)}")
print(f"escala_grises: {escala_grises}")
print(f"n_apilados: {config.get('n_apilados', 4)}")
print(f"pasos entrenados: {modelo.num_timesteps}")

## Evaluacion greedy

Politica greedy (`deterministic=True`), recompensa real del juego sin clipping, 5 episodios. Se reporta el maximo y la media, como en la competencia.

In [ ]:
puntajes = evaluar.evaluar(
    modelo,
    n_episodios=N_EPISODIOS,
    semilla=SEMILLA,
    escala_grises=escala_grises,
)

for i, puntaje in enumerate(puntajes):
    print(f"episodio {i}: {puntaje:.0f} puntos")
print(f"max {max(puntajes):.0f} / media {np.mean(puntajes):.1f}")

## Video del agente

Graba `EPISODIOS_VIDEO` episodios. `generar_video` omite grabar el episodio 0, por eso se piden 3 para garantizar al menos un video.

In [ ]:
os.makedirs(carpeta_video, exist_ok=True)
rutas = evaluar.generar_video(
    modelo,
    carpeta_video,
    n_episodios=EPISODIOS_VIDEO,
    semilla=SEMILLA,
    escala_grises=escala_grises,
)
for ruta in rutas:
    print(ruta)

## Video embebido

Ultimo video grabado, embebido para reproducirlo dentro del notebook durante la presentacion.

In [ ]:
Video(rutas[-1], embed=True)